# Preparation
Import and load a dataloader from the default config

In [ ]:
import os
import hydra
from hydra import initialize, compose
from rich.pretty import pprint
import numpy as np

os.environ["PROJECT_ROOT"] = "../"
hdf5_name = "Aq1T0_270z_770z_2.h5"
def load_augmentation(augmentation_name=None, batch_size=4, crop_size=64):
    with initialize(version_base="1.3", config_path="../configs", job_name="notebook_visualization"):
        if augmentation_name is not None:
            cfg = compose(config_name="train.yaml", overrides=["task_name=augmentations", f"crop_size={crop_size}", f"data.batch_size={batch_size}", f"data/augmentation/transforms=[{augmentation_name}]", f"hdf5_name=crops/hdf5/single/{hdf5_name}"])
        else: # default
            cfg = compose(config_name="train.yaml", overrides=["task_name=augmentations", f"crop_size={crop_size}", f"data.batch_size={batch_size}", f"hdf5_name=crops/hdf5/single/{hdf5_name}"])
                
        # Set augmentation probability to 1.0
        augmentations = cfg["data"]["augmentation"]["transforms"]
        for augmentation in augmentations:
            print(f"Setting config data.augmentation.transforms.{augmentation}.prob = 1.0")
            if cfg["data"]["augmentation"]["transforms"][augmentation].get("prob"):
                cfg["data"]["augmentation"]["transforms"][augmentation]["prob"] = 1.0
        
        pprint(cfg)
        
        augmentations = hydra.utils.instantiate(cfg.data.augmentation)               
        return augmentations

def load_config(batch_size=4, crop_size=64):
    with initialize(version_base="1.3", config_path="../configs", job_name="notebook_visualization"):
        cfg = compose(config_name="train.yaml", overrides=["task_name=augmentations", f"crop_size={crop_size}", f"data.batch_size={batch_size}", f"hdf5_name=crops/hdf5/single/{hdf5_name}"])
        pprint(cfg)        
        train_data = hydra.utils.instantiate(cfg.data.train_data)
        train_data.init() 
               
        return train_data

def augment(x, y, aug):
    augmented = aug({"image": x, "label": y})
    x, y = augmented["image"], augmented["label"]
    return x, y
    
def get_n_augment(data, aug, n):
    x_batch = []
    y_batch = []
    for i in range(n):
        # Get non augmented data
        x, y, _ = data[i]
        x_batch.append(x)
        y_batch.append(y)
        # Get augmented data
        xa, ya = augment(x, y, aug)
        x_batch.append(xa)
        y_batch.append(ya)
        
    return np.asarray(x_batch), np.asarray(y_batch)
    

In [2]:
# Create a display function
import napari

def display_item(x, y, aug_name):
    viewer = napari.Viewer()
    batch_size = x.shape[0]
    
    for i in range(batch_size):
        name = f"normal"
        index = np.floor(i/2)
        if i%2 == 1:
            name = f"{aug_name}"
        print(f"X range = {x.min()};{x.max()}")
        viewer.add_image(x[i, 0], name=f'im_{index}_{name}', colormap='gray', contrast_limits=[0, 1.0])
        viewer.add_image(y[i, 0], name=f'label_{index}_{name}', colormap='magma', contrast_limits=[0, 1.0])
    viewer.show()

In [ ]:
crop_size = 128
data = load_config(crop_size=crop_size)

{'task_name': 'augmentations', 'tags': ['dev'], 'train': True, 'test': True, 'ckpt_path': None, 'seed': 12345, 'date': '${now:%Y-%m-%d}_${now:%H-%M-%S-%f}', 'hdf5_name': 'crops/hdf5/single/Aq1T0_270z_770z_2.h5', 'crop_size': 64, 'window_overlap': 0.5, 'max_steps_per_epoch': 1000, 'monitor_val': 'val/loss', 'monitor_direction': 'min', 'extras': {'print_config': True}, 'benchmark_cuddn': True, 'find_lr': False, 'paths': {'root_dir': '${oc.env:PROJECT_ROOT}', 'data_dir': '${paths.root_dir}/data/', 'log_dir': '${paths.root_dir}/logs/', 'output_dir': '${hydra:runtime.output_dir}', 'work_dir': '${hydra:runtime.cwd}'}, 'model': {'_target_': 'ctforams.model.module.CTForamsLitModule', 'compile': False, 'scheduler_interval': 'step', 'criterion_on_center': False, 'criterion': {'_target_': 'monai.losses.GeneralizedDiceFocalLoss', 'include_background': True, 'to_onehot_y': False, 'sigmoid': True}, 'optimizer': {'_target_': 'torch.optim.AdamW', '_partial_': True, 'lr': 0.001, 'betas': [0.9, 0.999], 'weight_decay': 0.0001}, 'scheduler': {'_target_': 'torch.optim.lr_scheduler.OneCycleLR', '_partial_': True, 'max_lr': '${model.optimizer.lr}', 'epochs': '${trainer.max_epochs}', 'steps_per_epoch': None, 'pct_start': 0.1}, 'net': {'_target_': 'segmentation_models_pytorch_3d.Unet', 'encoder_name': 'efficientnet-b0', 'encoder_weights': None, 'in_channels': 1, 'classes': 1}}, 'data': {'train_data': {'_target_': 'ctforams.data.dataset.ctforams.CTForamsDataset', 'hdf5_path': '${paths.data_dir}/${hdf5_name}', 'group_name': 'train', 'crop_size': '${crop_size}', 'max_steps_per_epoch': '${max_steps_per_epoch}', 'window_overlap': '${window_overlap}', 'training': True}, 'val_data': {'_target_': 'ctforams.data.dataset.ctforams.CTForamsDataset', 'hdf5_path': '${paths.data_dir}/${hdf5_name}', 'group_name': 'val', 'crop_size': '${crop_size}', 'max_steps_per_epoch': '${max_steps_per_epoch}', 'window_overlap': 0.0}, 'test_data': {'_target_': 'ctforams.data.dataset.ctforams.CTForamsDataset', 'hdf5_path': '${paths.data_dir}/${hdf5_name}', 'group_name': 'test', 'crop_size': '${crop_size}', 'max_steps_per_epoch': '${max_steps_per_epoch}', 'window_overlap': 0.0}, 'augmentation': {'transforms': {'flip': {'_target_': 'monai.transforms.RandFlipd', 'keys': ['image', 'label'], 'prob': 0.5}, 'affine': {'_target_': 'monai.transforms.RandAffined', 'keys': ['image', 'label'], 'prob': 0.5, 'mode': ['bilinear', 'bilinear'], 'spatial_size': ['${crop_size}', '${crop_size}', '${crop_size}'], 'rotate_range': [3.14, 3.14, 3.14], 'scale_range': [0.0, 0.0, 0.0]}, 'random_noise': {'_target_': 'ctforams.data.augmentation.AugmentationWrapper', 'transforms': {'histogram_shift': {'_target_': 'monai.transforms.RandHistogramShiftd', 'keys': ['image'], 'prob': 0.5, 'num_control_points': 3}, 'intensity_shift': {'_target_': 'monai.transforms.RandShiftIntensityd', 'keys': ['image'], 'offsets': [-0.3, 0.3], 'safe': True, 'prob': 0.5, 'channel_wise': False}, 'adjust_contrast': {'_target_': 'monai.transforms.RandAdjustContrastd', 'keys': ['image'], 'prob': 0.5, 'gamma': [0.0, 2], 'retain_stats': False}, 'gaussian_noise': {'_target_': 'monai.transforms.RandGaussianNoised', 'keys': ['image'], 'prob': 0.5, 'mean': 0.0, 'std': 0.04}, 'gaussian_smooth': {'_target_': 'monai.transforms.RandGaussianSmoothd', 'keys': ['image'], 'prob': 0.5, 'sigma_x': [0.25, 1], 'sigma_y': [0.25, 1], 'sigma_z': [0.25, 1]}}, 'target': 'oneof'}}, '_target_': 'ctforams.data.augmentation.AugmentationWrapper'}, '_target_': 'ctforams.data.datamodule.CTForamsDataModule', 'batch_size': 4, 'num_workers': 0, 'pin_memory': False, 'crop_size': '${crop_size}'}, 'callbacks': {'model_checkpoint': {'_target_': 'lightning.pytorch.callbacks.ModelCheckpoint', 'dirpath': '${paths.output_dir}/checkpoints', 'filename': 'best', 'monitor': '${monitor_val}', 'verbose': False, 'save_last': True, 'save_top_k': 1, 'mode': '${monitor_direction}', 'auto_insert_metric_name': False, 'save_weights_only': False, 'every_n_train_steps': None, 'train_time_inter

/home/rkarpins/miniconda3/envs/ctforams/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
35518it [00:36, 969.72it/s] 


In [ ]:
def show_aug(name=None):
    aug = load_augmentation(name, crop_size=crop_size)
    x, y = get_n_augment(data, aug, 1)
    display_item(x, y, name)

In [6]:
show_aug()

Setting config data.augmentation.transforms.flip.prob = 1.0
Setting config data.augmentation.transforms.affine.prob = 1.0
Setting config data.augmentation.transforms.random_noise.prob = 1.0


{'task_name': 'augmentations', 'tags': ['dev'], 'train': True, 'test': True, 'ckpt_path': None, 'seed': 12345, 'date': '${now:%Y-%m-%d}_${now:%H-%M-%S-%f}', 'hdf5_name': 'crops/hdf5/single/Aq1T0_270z_770z_2.h5', 'crop_size': 64, 'window_overlap': 0.5, 'max_steps_per_epoch': 1000, 'monitor_val': 'val/loss', 'monitor_direction': 'min', 'extras': {'print_config': True}, 'benchmark_cuddn': True, 'find_lr': False, 'paths': {'root_dir': '${oc.env:PROJECT_ROOT}', 'data_dir': '${paths.root_dir}/data/', 'log_dir': '${paths.root_dir}/logs/', 'output_dir': '${hydra:runtime.output_dir}', 'work_dir': '${hydra:runtime.cwd}'}, 'model': {'_target_': 'ctforams.model.module.CTForamsLitModule', 'compile': False, 'scheduler_interval': 'step', 'criterion_on_center': False, 'criterion': {'_target_': 'monai.losses.GeneralizedDiceFocalLoss', 'include_background': True, 'to_onehot_y': False, 'sigmoid': True}, 'optimizer': {'_target_': 'torch.optim.AdamW', '_partial_': True, 'lr': 0.001, 'betas': [0.9, 0.999], 'weight_decay': 0.0001}, 'scheduler': {'_target_': 'torch.optim.lr_scheduler.OneCycleLR', '_partial_': True, 'max_lr': '${model.optimizer.lr}', 'epochs': '${trainer.max_epochs}', 'steps_per_epoch': None, 'pct_start': 0.1}, 'net': {'_target_': 'segmentation_models_pytorch_3d.Unet', 'encoder_name': 'efficientnet-b0', 'encoder_weights': None, 'in_channels': 1, 'classes': 1}}, 'data': {'train_data': {'_target_': 'ctforams.data.dataset.ctforams.CTForamsDataset', 'hdf5_path': '${paths.data_dir}/${hdf5_name}', 'group_name': 'train', 'crop_size': '${crop_size}', 'max_steps_per_epoch': '${max_steps_per_epoch}', 'window_overlap': '${window_overlap}', 'training': True}, 'val_data': {'_target_': 'ctforams.data.dataset.ctforams.CTForamsDataset', 'hdf5_path': '${paths.data_dir}/${hdf5_name}', 'group_name': 'val', 'crop_size': '${crop_size}', 'max_steps_per_epoch': '${max_steps_per_epoch}', 'window_overlap': 0.0}, 'test_data': {'_target_': 'ctforams.data.dataset.ctforams.CTForamsDataset', 'hdf5_path': '${paths.data_dir}/${hdf5_name}', 'group_name': 'test', 'crop_size': '${crop_size}', 'max_steps_per_epoch': '${max_steps_per_epoch}', 'window_overlap': 0.0}, 'augmentation': {'transforms': {'flip': {'_target_': 'monai.transforms.RandFlipd', 'keys': ['image', 'label'], 'prob': 1.0}, 'affine': {'_target_': 'monai.transforms.RandAffined', 'keys': ['image', 'label'], 'prob': 1.0, 'mode': ['bilinear', 'bilinear'], 'spatial_size': ['${crop_size}', '${crop_size}', '${crop_size}'], 'rotate_range': [3.14, 3.14, 3.14], 'scale_range': [0.0, 0.0, 0.0]}, 'random_noise': {'_target_': 'ctforams.data.augmentation.AugmentationWrapper', 'transforms': {'histogram_shift': {'_target_': 'monai.transforms.RandHistogramShiftd', 'keys': ['image'], 'prob': 0.5, 'num_control_points': 3}, 'intensity_shift': {'_target_': 'monai.transforms.RandShiftIntensityd', 'keys': ['image'], 'offsets': [-0.3, 0.3], 'safe': True, 'prob': 0.5, 'channel_wise': False}, 'adjust_contrast': {'_target_': 'monai.transforms.RandAdjustContrastd', 'keys': ['image'], 'prob': 0.5, 'gamma': [0.0, 2], 'retain_stats': False}, 'gaussian_noise': {'_target_': 'monai.transforms.RandGaussianNoised', 'keys': ['image'], 'prob': 0.5, 'mean': 0.0, 'std': 0.04}, 'gaussian_smooth': {'_target_': 'monai.transforms.RandGaussianSmoothd', 'keys': ['image'], 'prob': 0.5, 'sigma_x': [0.25, 1], 'sigma_y': [0.25, 1], 'sigma_z': [0.25, 1]}}, 'target': 'oneof'}}, '_target_': 'ctforams.data.augmentation.AugmentationWrapper'}, '_target_': 'ctforams.data.datamodule.CTForamsDataModule', 'batch_size': 4, 'num_workers': 0, 'pin_memory': False, 'crop_size': '${crop_size}'}, 'callbacks': {'model_checkpoint': {'_target_': 'lightning.pytorch.callbacks.ModelCheckpoint', 'dirpath': '${paths.output_dir}/checkpoints', 'filename': 'best', 'monitor': '${monitor_val}', 'verbose': False, 'save_last': True, 'save_top_k': 1, 'mode': '${monitor_direction}', 'auto_insert_metric_name': False, 'save_weights_only': False, 'every_n_train_steps': None, 'train_time_inter

X range = 0.05098039284348488;0.9529411792755127
X range = 0.05098039284348488;0.9529411792755127


In [ ]:
show_aug("monai_flip")

In [ ]:
show_aug("monai_affine")

In [ ]:
show_aug("monai_adjust_contrast")

In [ ]:
show_aug("monai_elastic")

In [ ]:
show_aug("monai_gaussian_noise")

In [ ]:
show_aug("monai_gaussian_smooth")

In [ ]:
show_aug("monai_grid_distortion")

In [ ]:
show_aug("monai_histogram_normalized")

In [ ]:
show_aug("monai_histogram_shift")

In [ ]:
show_aug("monai_intensity_shift")

In [ ]:
show_aug("monai_simulate_low_res")